# KG catch-up + intent chat session -> turns + knowledge graph

A new process flow on top of **[`kg_intent_chat.ipynb`](kg_intent_chat.ipynb)**: instead of
starting cold, the agent first figures out **how long it's been since it last spoke with this
human, what they'd talked about back then, and how MUCH of that it should expect to hear about
again** -- then keeps the conversation going, topic by topic, until it judges it has *enough* for
the catch-up period, not just "asked once about everything."

Built on **[`gaps_from_kg/get_temporal_containers.py`](../src/cltl/gaps_from_kg/get_temporal_containers.py)**
-- a different query layer over the same GraphDB repository (via `cltl.brain.LongTermMemory`)
than `kg_gap_finder.py`/`intent_gap_finder.py`'s rdflib/SPARQL-endpoint queries, which is what the
rest of the per-turn flow below still uses unchanged. See
**[`catch_up_from_kg.py`](catch_up_from_kg.py)** for the full implementation; in short, a loop:

1. **Ask about a catch-up-period topic** (`SaturationTracker.next_question()`) -- one of
   `chat_sessions.DEFAULT_GAP_ACTIVITY_TYPES` (exercise, diet, sleep, symptoms, medication, ...)
   this human has real history with, but not yet ENOUGH reported for this period (see
   "Saturation" below).
2. **Whatever they report is handled entirely by `KgIntentChatSession`'s own EXISTING per-turn
   flow, unchanged**: SRL extraction -> push to the KG -> `intent_gap_finder.next_intent_gap()`
   keeps asking its own follow-up questions about *that* activity (what/how much/when/where) for
   as long as its matching intent still has unmet requirements.
3. **Once that activity's own follow-ups are exhausted**, go back to step 1 -- another
   under-covered topic, or the same one again if it's still short -- **unless every topic has
   reached saturation**, at which point (see "Wrapping up" below) the LLM decides how to close
   things out, and the conversation continues normally after that.

**The catch-up period is capped at `catch_up.MAX_SATURATION_GAP_DAYS` (14 days)**, even when the
real gap since the last conversation is much longer -- a human who hasn't talked in two months
still only needs the last two weeks caught up on live, not the whole two months.

**Saturation** -- "enough knowledge for the catch-up period" -- is defined *per topic* against a
fixed **weekly** baseline (`catch_up.CATCH_UP_WINDOW_DAYS`, 7 days), not one tied to however long
the catch-up period itself is: `_windowed_average_rate()` tiles that week-long window backwards
across this human's *entire* history to get their typical weekly rate, which is then scaled to
however many days the (capped) catch-up period covers. If they've historically reported exercise
~3 times a week and the catch-up period is 2 weeks, 6 reported live is enough for exercise
specifically -- not "ask about it once and move on" and not "keep asking forever." A hard
per-topic cap (`SaturationTracker.MAX_ASKS_PER_TOPIC`, default 3) still gives up on a topic the
human simply has nothing more to say about, regardless of its target.

**Wrapping up.** The FIRST time every topic is saturated (or capped out),
`SaturationTracker.wrap_up_message()` sends one message naming what was actually covered and
lets the LLM decide for itself: continue with a short question if there's an obvious thread left,
or wrap up warmly and say goodbye. After that one message, replies fall through to the plain
default `agent_fn` unchanged, so the conversation continues (or ends, if the human says goodbye
back) like an ordinary chat.

**Before running this:** same requirements as `kg_intent_chat.ipynb` -- `OPENAI_API_KEY` set, and
a reachable knowledge-graph SPARQL endpoint (e.g. a local GraphDB `event_sandbox` repository) at
`KG_ADDRESS` below, ideally one that already has some history for `HUMAN` in it (otherwise there's
nothing to catch up on, and this degrades to a plain, short "what's new?" opener with an
immediately-saturated tracker).

In [1]:
# In cltl-kg-driven-chat, chat_sessions.py and catch_up_from_kg.py sat next to these notebooks;
# in this project they live in src/kg_chat, and import each other by bare name. src/ comes first
# as well, so that src/cltl/commons replaces the installed cltl.commons (see README.md).
# Run the notebooks from notebooks/: src/cltl and intents/ are found from there.
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
for _dir in (str(_SRC / "kg_chat"), str(_SRC)):
    if _dir not in sys.path:
        sys.path.insert(0, _dir)

In [2]:
import time
from datetime import datetime, timedelta

from chat_sessions import KgIntentChatSession, openai_agent, save_turns
import catch_up_from_kg as catch_up

KG_ADDRESS = "http://localhost:7200/repositories/event_sandbox"
KG_LOG_DIR = "kg_logs"
HUMAN = "Mehmet"
HUMAN = "Jan"

# Directory of intent *.json files -- None auto-discovers the project's own intents/ folder (see
# intent_gap_finder.load_intents()/_default_intents_dir()).
INTENTS_DIR = None

# A FRESH id every run, not a fixed constant -- see kg_intent_chat.ipynb's own CHAT_ID cell for
# why (reusing one across runs collides every run's activities on the same subject URIs).
CHAT_ID = int(time.time())

# "Now", for both the catch-up queries below and every activity pushed to the graph this run.
CURRENT_DATE = datetime.now()

# Used only if the KG has no earlier conversation with HUMAN on record at all (e.g. a brand new
# human, or a fresh/empty graph) -- how far back to assume the last (nonexistent) conversation
# was, so there's still a sensible "it's been N days" opener instead of a crash.
FALLBACK_LAST_CONVERSATION_DATE = CURRENT_DATE - timedelta(days=7)


## Step 1: how long has it been, and what's worth catching up on?

Connects to the same KG `KG_ADDRESS` points at (via `cltl.brain.LongTermMemory` this time, not
rdflib). `connect_brain()` also makes sure the graph has
**[`gaps_from_kg/n2mu_sem_roles.py`](../src/cltl/gaps_from_kg/n2mu_sem_roles.py)**'s small
`rdfs:subPropertyOf` mapping uploaded (a one-time, idempotent step -- see that module's own
docstring): without it, every real activity looks dateless/actorless/placeless to
`get_temporal_containers()`'s underlying query, since activities only ever carry this project's
own `n2mu:agent`/`n2mu:location`/`n2mu:time/...` predicates, never `sem:hasActor`/`hasPlace`/
`hasTime` directly. Past that one-time schema write, the rest of this step only reads.

`find_catch_up_topics()` caps the period it actually tries to catch up on at
`MAX_SATURATION_GAP_DAYS` days (see the intro above), and calibrates each topic's target against
a fixed *weekly* baseline (`weekly_rate`) computed from this human's full history before that
period -- not against the (possibly much longer) real gap itself.

In [3]:
brain = catch_up.connect_brain(KG_ADDRESS, log_dir=KG_LOG_DIR)

last_conversation_date = catch_up.find_last_conversation_date(
    HUMAN, brain, CURRENT_DATE, FALLBACK_LAST_CONVERSATION_DATE
)
catch_up_topics = catch_up.find_catch_up_topics(brain, CURRENT_DATE, last_conversation_date)

gap_days = (CURRENT_DATE.date() - last_conversation_date.date()).days
capped_days = min(gap_days, catch_up.MAX_SATURATION_GAP_DAYS)
print(f"Last conversation with {HUMAN}: {last_conversation_date} ({gap_days} day(s) ago)")
print(f"Catching up on the last {capped_days} day(s)"
      + (f" (capped from the full {gap_days}-day gap)" if capped_days < gap_days else ""))
print(f"Found {len(catch_up_topics)} catch-up topic(s):")
for topic in catch_up_topics:
    print(f"  - {topic['activity_type']}: {topic['weekly_rate']:.1f}/week historically -> "
          f"target {topic['expected_count']} for this period (already "
          f"{topic['initial_reported_count']} logged), most recently "
          f"\"{topic['latest_label']}\" on {topic['latest_date']}")


2026-10-06 09:27:23 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-10-06 09:27:23 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-10-06 09:27:23 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-10-06 09:27:23 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-10-06 09:27:23 -     INFO -                                   cltl.brain.TrustCalculator - Booted


Our previous conservation was on Tuesday 2026-09-29 09:27:16.378016
Today is Tuesday 2026-10-06 09:27:16.378016
What happened in the last 7 days?
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:exercise.                 ?id rdfs:label ?label .                 }
I found 35 activities
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:take_food.                 ?id rdfs:label ?label .                 }
I found 23 activities
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:take_drink.                 ?id rdfs:label ?label .                 }
I found 46 activities
PREFIX n2mu: <http://cltl.nl/leo

## Step 2: open the chat

`SaturationTracker.opening_question()` turns the findings above into the agent's very first turn
(recorded via `open_with()`, same as `kg_chat_session.ipynb`'s own opening-greeting cell -- see
there for why it's a real, annotated-and-pushed turn rather than just printed text), in this
order: (1) how long it's been since the last conversation, (2) a one-sentence summary of EVERY
topic talked about back then (`catch_up_topics`, this human's own full list, not just a couple of
"lead" ones), and only then (3) a single, specific question about whichever ONE topic most needs
catching up on -- marked as asked once so the loop below doesn't immediately repeat it.

`wrap_agent_fn_with_saturation_loop()` then wraps the plain `openai_agent()` default reply
function: for as long as `tracker.is_saturated()` is False, a reply that would otherwise be the
generic default LLM one asks about the next under-covered topic instead -- see
`catch_up_from_kg.py`'s own module docstring for exactly when that fires versus an ordinary
intent-driven gap question, and how `on_new_subject=tracker.record_new_activity` keeps the
tracker's own counts up to date as the human reports things live.

In [4]:
from kg_chat_gui import run_gui
from kg_question_answerer import KgQuestionAnswerer

tracker = catch_up.SaturationTracker(catch_up_topics, human=HUMAN)
agent_fn = catch_up.wrap_agent_fn_with_saturation_loop(openai_agent(), tracker)

kg_session = KgIntentChatSession(
    chat=CHAT_ID,
    human=HUMAN,
    kg_address=KG_ADDRESS,
    log_dir=KG_LOG_DIR,
    intents_dir=INTENTS_DIR,
    agent_fn=agent_fn,
    on_new_subject=tracker.record_new_activity,
    # Answers the human's questions about their own record ("what did I drink?") from the
    # knowledge graph, instead of extracting the question as a new activity.
    question_answerer=KgQuestionAnswerer(KG_ADDRESS, HUMAN),
)
print(f"Loaded {len(kg_session.intents)} intent(s) covering activity types: "
      f"{sorted({t for intent in kg_session.intents for t in intent.get('activity_types', [])})}")

opening_question = tracker.opening_question(CURRENT_DATE, last_conversation_date)
print("saturation targets:", {t: (tracker.reported[t], target["expected_count"]) for t, target in tracker.targets.items()})

kg_session.open_with(opening_question)
kg_turns = run_gui(kg_session)


Loaded 17 intent(s) covering activity types: ['economic_condition', 'exercise', 'measurement', 'mental_condition', 'physical_condition', 'sleep', 'social_condition', 'symptom', 'take_drink', 'take_food', 'take_medicine']


2026-10-06 09:27:42 -     INFO -                                                       httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


saturation targets: {'physical_condition': (2, 1), 'medication': (3, 1), 'take_food': (7, 2), 'treatment': (4, 2), 'take_drink': (9, 7), 'exercise': (2, 1), 'symptom': (2, 1), 'mental_condition': (6, 1), 'diet': (0, 1), 'measurement': (1, 1), 'sleep': (2, 2), 'social_condition': (0, 1)}
turn {'chat': 1791271636, 'human': 'Jan', 'date': '2026,Oct,06', 'turn': 1, 'speaker': 'agent', 'utterance': 'Hi Jan, it’s been about a week since we last talked. Last time we checked in on how you were feeling physically, your meds and treatment, what you eat and drink, your exercise, symptoms, mood, blood sugar measurements, sleep, and social life. Since then, what’s changed—if anything—in what, when, or how you’ve been eating?'}


2026-10-06 09:27:45 -     INFO -                                                       httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 09:27:45 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-10-06 09:27:45 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-10-06 09:27:45 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-10-06 09:27:45 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-10-06 09:27:45 -     INFO -                                   cltl.brain.TrustCalculator - Booted


Compliance issues for chat 1791271636 turn 1:
 - extraction[0]: activity offset auto-corrected for 'eating': (270,6) -> (316,6)
 - extraction[0]: agent_patient offset auto-corrected for 'you': (261,3) -> (84,3)
Total nr of scenarios 1


...

2026-10-06 09:30:19 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: meals_agent_patient_Jan [activity or take_food_->_person])
2026-10-06 09:30:19 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: meals_time_in the last week or two [activity or take_food_->_range])


chat 1791271636 out of  1 turn 13 out of 1 turns


100%|██████████| 1/1 [00:00<00:00,  4.27it/s]

[turn 13] agent: What meals have you eaten in the last week or two?
    pushed 2 triple(s):
      meals  agent_patient  =  you
      meals  time  =  in the last week or two


[kg_chat_gui] conversation saved to /Users/piek/Desktop/Leolani/cltl-apps/cltl-custom-diabetes/notebooks/chat_logs/chat1791271636_turns_20261006-093024.json
[kg_chat_gui] statistics saved to  /Users/piek/Desktop/Leolani/cltl-apps/cltl-custom-diabetes/notebooks/chat_logs/chat1791271636_stats_20261006-093024.json
[kg_chat_gui] gap log saved to     /Users/piek/Desktop/Leolani/cltl-apps/cltl-custom-diabetes/notebooks/chat_logs/chat1791271636_gaplog_20261006-093024.json


Inspect what was extracted, pushed, and where each agent reply came from:

In [4]:
print(f"{len(kg_session.turns)} turns, {len(kg_session.annotations)} annotated, "
      f"{len(kg_session.kg_pushes)} pushes to the knowledge graph")
print("reply sources:", kg_session.reply_sources)
print("saturated:", tracker.is_saturated(), "| wrapped up:", tracker.wrapped_up)
print("reported vs. target, per topic:",
      {t: (tracker.reported[t], target["expected_count"]) for t, target in tracker.targets.items()})
print("catch-up questions actually asked:", tracker.asked_log)
kg_session.kg_pushes


29 turns, 29 annotated, 29 pushes to the knowledge graph
reply sources: ['gap', 'gap', 'gap', 'gap', 'gap', 'gap', 'default', 'gap', 'gap', 'gap', 'default', 'default', 'default', 'default']
saturated: True | wrapped up: True
reported vs. target, per topic: {'physical_condition': (3, 1), 'take_drink': (8, 7), 'mental_condition': (4, 1), 'measurement': (2, 1), 'sleep': (3, 2), 'exercise': (2, 1), 'diet': (1, 1), 'take_food': (4, 2), 'treatment': (3, 2), 'symptom': (2, 1)}
catch-up questions actually asked: [{'activity_type': 'take_drink', 'attempt': 1}, {'activity_type': 'take_drink', 'attempt': 2}]


[{'conversations': 1, 'capsules': 8},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 3},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 3},
 {'conversations': 1, 'capsules': 2},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 3},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 2},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversat

`kg_session.turn_log` has the same per-turn breakdown `kg_intent_chat.ipynb` prints live: what
each turn pushed, which intent (if any) matched, and which requirement its reply was about
(`None` for a default- or saturation-loop-driven reply -- both are indistinguishable from
`turn_log`'s own point of view, since neither comes from `intent_gap_finder`;
`kg_session.reply_sources`/`tracker.asked_log` above are what tell them apart).

In [5]:
kg_session.turn_log

[{'turn': 1,
  'speaker': 'agent',
  'utterance': 'Hi Jan, it’s been about a week since we last talked. Last time we checked in on how you were feeling physically, your meds and treatment, what you eat and drink, your exercise, symptoms, mood, blood sugar measurements, sleep, and social life. Since then, what’s changed—if anything—in what, when, or how you’ve been eating?',
  'triples_pushed': [{'subject': 'eating',
    'predicate': 'agent_patient',
    'object': 'you'}],
  'gap_queries': [],
  'selected_gap': None},
 {'turn': 2,
  'speaker': 'Jan',
  'utterance': 'How did I sleep?',
  'triples_pushed': [],
  'gap_queries': [],
  'selected_gap': None},
 {'turn': 3,
  'speaker': 'agent',
  'utterance': 'From what you’ve told me, you normally sleep about 6 hours a night. On Sunday 20 September you slept about 12 hours, which was a big change, and around then you were still feeling tired and sometimes fell asleep watching TV. Over the three days leading up to 26 September, you said you we

Save the turns, plus an **intent log** under `notebooks/intents_log/` summarizing the whole
session: the gap and the topics identified before the chat began (`topics_at_start`), how each of
those topics actually fared live (`topics_covered` -- reported/asked counts, whether its target
was met or it was capped out, and the tracker's own `asked_log`), and every
`intent_gap_finder.py` intent the per-turn flow consulted along the way (`intents_covered`) --
see `catch_up_from_kg.save_intent_log()` for the exact shape.

In [6]:
save_turns(kg_session.turns, "intents_log/kg_catchup_intent_turns.json")
catch_up.save_intent_log(kg_session, tracker, catch_up_topics, CURRENT_DATE, last_conversation_date)


Wrote 13 turns to intents_log/kg_catchup_intent_turns.json
Wrote intent log to intents_log/chat1791271636_intents_20261006-093053.json


PosixPath('intents_log/chat1791271636_intents_20261006-093053.json')